# Step 2a: Data Cleaning - Python Baseline Preprocessing for Publishers

## Overview
This notebook implements syntactic cleaning for publisher names extracted from MARC authority-controlled fields (100, 110, 700, 710, 720). The goal is to standardize publisher names for accurate aggregation in downstream visualization (e.g. Treemap, Timeline).

**Input**: `data/raw/publishers.csv` (376 unique publishers)  
**Output**: `data/raw/publishers_cleaned.csv` (dual-column: raw + cleaned)  
**Next Step**: Step 3 (Merge with book_publishers bridge table; skip OpenRefine)

## Setup: Imports and Configuration

In [7]:
import pandas as pd
import re
import os

# Configure pandas display
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Load Input Data

In [8]:
# Load publishers from Step 1 extraction
df_publishers = pd.read_csv('../data/raw/publishers.csv')

# Rename original column to preserve raw values
df_publishers.rename(columns={'publisher_name': 'publisher_name_raw'}, inplace=True)

print(f"✓ Loaded publishers.csv: {len(df_publishers)} rows")
print(f"\nColumn structure:")
print(df_publishers.head(10))

✓ Loaded publishers.csv: 376 rows

Column structure:
                                   publisher_name_raw
0                                    Vleminckx, Henri
1        E. Plon, Nourrit et Cie, imprimeurs-éditeurs
2                            Buschmann, Joseph-Ernest
3                             Marchands de nouveautés
4                                  Librairie Nouvelle
5                   El Mensajero del corazon de Jesus
6                                Jouret et Thémon, E.
7  Kommissionsverlag der Fr. Lintz'schen Buchhandlung
8                          C. Marpon et E. Flammarion
9                                       Perrin et Cie


## Define Publisher Cleaning Function

Multi-stage syntactic normalization for historical MARC publisher names

In [9]:
def clean_publisher(pub_raw):
    """
    Step 2a: Syntactic cleaning for MARC authority-controlled publisher names.
    
    Processing stages:
    1. NULL/NaN handling
    2. Removal of [s.n.] (sine nomine - no publisher)
    3. Removal of outer double quotation marks
    4. Removal of bracketed content (supplementary info: names, titles, roles)
    5. Removal of occupational suffixes (imprimeur, libraire, éditeur)
    6. Replacement of 'et' with '&' for consistency
    7. Removal of trailing punctuation
    8. Whitespace normalization
    9. Title Case formatting
    10. Final validation
    
    Args:
        pub_raw: Raw publisher name string from MARC fields
        
    Returns:
        str or None: Cleaned publisher name, or None for [s.n.] entries
    """
    
    # Stage 1: Handle NULL values
    if pd.isna(pub_raw):
        return None
    
    pub = str(pub_raw).strip()
    
    # Stage 2: Remove [s.n.] and s.n. (sine nomine - no publisher name)
    if pub in ['[s.n.]', 's.n.', '[s.n]', 's.n']:
        return None
    
    # Stage 3.1: Remove outer double quotation marks
    pub = pub.strip('"').strip()

    # Stage 3.2: Remove all double quotation marks
    pub = pub.replace('"', '')
    
    # Stage 4.1: Remove bracketed content [supplementary information]
    # Examples: [Eugène], [Imprimerie], [J.], [Librairie européenne]
    pub = re.sub(r'\s*\[.*?\]\s*', ' ', pub)

    # Stage 4.2: Remove bracketed content [supplementary information]
    pub = re.sub(r'\s*\(.*?\)\s*', ' ', pub)

    # Stage 5: Remove occupational suffixes at the end of string
    # These describe the publisher's role but are not part of the official name
    pub = re.sub(
    r',\s*(imprimeur|editeur|libraire|printer|publisher|imprimerie)s?\b.*$',
    '', pub, flags=re.IGNORECASE
    )
    
    # Stage 6: Replace ' et ' (French 'and') with ' & ' for standardization
    # This normalizes inconsistent usage: "Perrin et Cie" → "Perrin & Cie"
    pub = re.sub(r'\s+et\s+', ' & ', pub, flags=re.IGNORECASE)
    
    # Stage 7: Remove trailing punctuation expect for "."
    pub = pub.rstrip(',;:-')
    
    # Stage 8: Normalize whitespace (collapse multiple spaces)
    pub = re.sub(r'\s+', ' ', pub).strip()
    
    # Stage 9: Apply Title Case
    pub = pub.title()
    
    # Stage 10: Final validation - reject empty strings
    if pub == "" or pub.isspace():
        return None
    
    return pub

print("✓ Publisher cleaning function defined")

✓ Publisher cleaning function defined


## Apply Cleaning

In [10]:
# Apply cleaning function to all publishers
df_publishers['publisher_name_cleaned'] = df_publishers['publisher_name_raw'].apply(clean_publisher)
df_publishers['publisher_name_cleaned'] = df_publishers['publisher_name_cleaned'].fillna('Unknown')
df_publishers['publisher_name_cleaned'] = df_publishers['publisher_name_cleaned'].replace('', 'Unknown')

# Count how many entries were removed (converted to None)
unknown_count = (df_publishers['publisher_name_cleaned'] == 'Unknown').sum()

print(f"\n=== CLEANING EXECUTION ===")
print(f"Input rows: {len(df_publishers)}")
print(f"Entries identified as Unknown: {unknown_count}")
print(f"Entries retained: {len(df_publishers) - unknown_count}")


=== CLEANING EXECUTION ===
Input rows: 376
Entries identified as Unknown: 2
Entries retained: 374


## Data Quality Validation

In [11]:
print("\n=== DATA QUALITY VALIDATION ===")

# 1. NULL value check
print(f"\nNULL value analysis:")
print(f"  publisher_name_raw NULL values: {df_publishers['publisher_name_raw'].isna().sum()}")
print(f"  publisher_name_cleaned NULL values: {df_publishers['publisher_name_cleaned'].isna().sum()}")
print(f"  → All NULL values are intentional [s.n.] removals ✓")

# 2. Check for remaining problematic characters
print(f"\nRemaining character check:")
df_valid = df_publishers[df_publishers['publisher_name_cleaned'].notna()]

remaining_quotes = df_valid['publisher_name_cleaned'].str.contains('"').sum()
remaining_brackets = df_valid['publisher_name_cleaned'].str.contains(r'\[').sum()
remaining_sn = df_valid['publisher_name_cleaned'].str.contains(r'\[s\.n\.\]', regex=True).sum()

print(f"  Entries with remaining quotes: {remaining_quotes}")
print(f"  Entries with remaining brackets: {remaining_brackets}")
print(f"  Entries with [s.n.]: {remaining_sn}")

if remaining_quotes == 0 and remaining_brackets == 0 and remaining_sn == 0:
    print(f"  ✓ All checks passed - no problematic characters remain")
else:
    print(f"  ⚠ Warning: Issues detected above")


=== DATA QUALITY VALIDATION ===

NULL value analysis:
  publisher_name_raw NULL values: 0
  publisher_name_cleaned NULL values: 0
  → All NULL values are intentional [s.n.] removals ✓

Remaining character check:
  Entries with remaining quotes: 0
  Entries with remaining brackets: 0
  Entries with [s.n.]: 0
  ✓ All checks passed - no problematic characters remain


## Cleaning Effectiveness Analysis

In [16]:
print("\n=== CLEANING EFFECTIVENESS ===")

# Count transformations
df_analysis = df_publishers[df_publishers['publisher_name_cleaned'].notna()].copy()

# Count specific transformations
et_replacements = df_analysis['publisher_name_raw'].str.contains(r'\s+et\s+', regex=True, case=False).sum()
bracket_removals = df_analysis['publisher_name_raw'].str.contains(r'\[.*?\]', regex=True).sum()
quote_removals = df_analysis['publisher_name_raw'].str.contains('"').sum()
role_suffix_removals = df_analysis['publisher_name_raw'].str.contains(
    r'imprimeur|libraire|éditeur|printer|publisher|imprimerie', 
    regex=True, case=False
).sum()
removed_count = df_analysis['publisher_name_raw'].str.contains(r'\[s\.n\.\]', regex=True, case=False).sum()

print(f"\nSpecific transformations applied:")
print(f"  'et' → '&' replacements: {et_replacements} entries")
print(f"  Bracket removals [content]: {bracket_removals} entries")
print(f"  Quote removals: {quote_removals} entries")
print(f"  Role suffix removals: {role_suffix_removals} entries")
print(f"  [s.n.] removals: {removed_count} entries")

# Deduplication check
unique_raw = df_analysis['publisher_name_raw'].nunique()
unique_cleaned = df_analysis['publisher_name_cleaned'].nunique()
duplicates_collapsed = unique_raw - unique_cleaned

print(f"\nDeduplication effectiveness:")
print(f"  Unique publishers (raw): {unique_raw}")
print(f"  Unique publishers (cleaned): {unique_cleaned}")
print(f"  Duplicates collapsed by standardization: {duplicates_collapsed}")
print(f"  Deduplication rate: {(duplicates_collapsed/unique_raw*100):.1f}%")


=== CLEANING EFFECTIVENESS ===

Specific transformations applied:
  'et' → '&' replacements: 37 entries
  Bracket removals [content]: 11 entries
  Quote removals: 1 entries
  Role suffix removals: 23 entries
  [s.n.] removals: 1 entries

Deduplication effectiveness:
  Unique publishers (raw): 376
  Unique publishers (cleaned): 374
  Duplicates collapsed by standardization: 2
  Deduplication rate: 0.5%


## Sample Inspection: Before/After Pairs

In [17]:
print("\n=== SAMPLE INSPECTION: Before/After Cleaning ===")

# Show entries where significant changes occurred
df_sample = df_publishers[
    (df_publishers['publisher_name_raw'] != df_publishers['publisher_name_cleaned']) &
    (df_publishers['publisher_name_cleaned'].notna())
].head(20)

print(f"\nEntries with significant transformations:")
for idx, row in df_sample.iterrows():
    print(f"\n  Raw:     {row['publisher_name_raw']}")
    print(f"  Cleaned: {row['publisher_name_cleaned']}")

# Show [s.n.] removals
df_sn = df_publishers[df_publishers['publisher_name_cleaned'].isna()]
print(f"\n\nEntries marked as [s.n.] (removed):")
for idx, row in df_sn.iterrows():
    print(f"  {row['publisher_name_raw']}")


=== SAMPLE INSPECTION: Before/After Cleaning ===

Entries with significant transformations:

  Raw:     E. Plon, Nourrit et Cie, imprimeurs-éditeurs
  Cleaned: E. Plon, Nourrit & Cie

  Raw:     Marchands de nouveautés
  Cleaned: Marchands De Nouveautés

  Raw:     El Mensajero del corazon de Jesus
  Cleaned: El Mensajero Del Corazon De Jesus

  Raw:     Jouret et Thémon, E.
  Cleaned: Jouret & Thémon, E.

  Raw:     Kommissionsverlag der Fr. Lintz'schen Buchhandlung
  Cleaned: Kommissionsverlag Der Fr. Lintz'Schen Buchhandlung

  Raw:     C. Marpon et E. Flammarion
  Cleaned: C. Marpon & E. Flammarion

  Raw:     Perrin et Cie
  Cleaned: Perrin & Cie

  Raw:     Dauvin et Fontaine
  Cleaned: Dauvin & Fontaine

  Raw:     Tipografia dentro la pietà de Turchini
  Cleaned: Tipografia Dentro La Pietà De Turchini

  Raw:     Fr. Beck's Universitäts-Buchhandlung
  Cleaned: Fr. Beck'S Universitäts-Buchhandlung

  Raw:     J. B. Istas, imprimeur-éditeur
  Cleaned: J. B. Istas

  Raw:     Bau

## Summary Statistics

In [18]:
print("\n=== STEP 2A SUMMARY ===")
print(f"\nPublishers:")
print(f"  Total input rows: {len(df_publishers)}")
print(f"  [s.n.] entries removed: {removed_count}")
print(f"  Retained for cleaning: {len(df_publishers) - removed_count}")
print(f"\nCleaning results:")
print(f"  Unique publishers (raw): {unique_raw}")
print(f"  Unique publishers (cleaned): {unique_cleaned}")
print(f"  Standardization deduplication: {duplicates_collapsed} duplicates collapsed")
print(f"\n✓ Step 2a preprocessing complete")
print(f"  Next: Step 2b skipped (no OpenRefine needed for aggregation-only visualizations)")
print(f"  Proceed: Step 3 (Merge with book_publishers bridge table)")


=== STEP 2A SUMMARY ===

Publishers:
  Total input rows: 376
  [s.n.] entries removed: 1
  Retained for cleaning: 375

Cleaning results:
  Unique publishers (raw): 376
  Unique publishers (cleaned): 374
  Standardization deduplication: 2 duplicates collapsed

✓ Step 2a preprocessing complete
  Next: Step 2b skipped (no OpenRefine needed for aggregation-only visualizations)
  Proceed: Step 3 (Merge with book_publishers bridge table)


## Export to CSV

In [ ]:
# Create output directory if it doesn't exist
output_dir = '../data/cleaned/'
os.makedirs(output_dir, exist_ok=True)

# Export dual-column mapping for Step 3
output_path = f'{output_dir}publishers_cleaned.csv'
df_publishers[['publisher_name_raw', 'publisher_name_cleaned']].to_csv(
    output_path, 
    index=False, 
    encoding='utf-8'
)

print(f"\n=== FILES EXPORTED ===")
print(f"✓ publishers_cleaned.csv")
print(f"  Location: {output_path}")
print(f"  Rows: {len(df_publishers)}")
print(f"  Columns: publisher_name_raw, publisher_name_cleaned")
print(f"\nReady for Step 3 (Merge + Aggregation for Treemap/Timeline visualization)")


=== FILES EXPORTED ===
✓ publishers_cleaned.csv
  Location: ../data/cleaned/publishers_cleaned.csv
  Rows: 376
  Columns: publisher_name_raw, publisher_name_cleaned

Ready for Step 3 (Merge + Aggregation for Treemap/Timeline visualization)
